# 02.2 — RAG and grounding lab

A complete retrieval-augmented generation loop, built by hand so nothing is magic:

1. Chunk a document three ways and measure the difference
2. Create an Azure AI Search index with a vector field and a semantic configuration
3. Embed and upload
4. Retrieve four ways — keyword, vector, hybrid, hybrid + semantic reranker
5. Ground a generation and produce checkable citations
6. Evaluate groundedness and retrieval
7. Add multistep reasoning: query rewrite, decomposition, self-check
8. Delete the index

> ⚠️ **Cost.** Azure AI Search bills **by the hour**, not by the query. The final
> cell deletes the index this lab creates, but that does **not** stop the service
> billing. Free tier is $0; Basic is roughly $75/month. If you provisioned Basic
> for this course, delete the service when you are done — see `99_teardown`.

**Prerequisites:** `AZURE_SEARCH_ENDPOINT` set; the Search service configured for
**RBAC** auth; you hold *Search Service Contributor* **and** *Search Index Data
Contributor*; `text-embedding-3-small` and `gpt-4o-mini` deployed.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import (
    cfg, credential, chat_client, project_client,
    search_index_client, search_client, ask, embed, show_usage,
)

INDEX_NAME = "ai103-rag-lab"   # created and deleted by this notebook

print("search    :", cfg.require("AZURE_SEARCH_ENDPOINT", unit="02.2"))
print("embedding :", cfg["MODEL_EMBEDDING"])
print("index     :", INDEX_NAME)

## 1. The corpus

Four short policy documents. Small enough to read, varied enough that retrieval
mode actually changes the answer — note the deliberate part numbers (`CX-4400`)
and the paraphrase-only overlaps.

In [ ]:
DOCS = {
    "returns-policy": {
        "title": "Contoso Returns Policy",
        "uri": "https://contoso.example/policies/returns",
        "text": """## Eligibility
Unopened items may be returned within 30 days of delivery for a full refund to the
original payment method. Proof of purchase is required.

## Opened items
Opened items are eligible for store credit only. Store credit is issued as a code
valid for 12 months and cannot be exchanged for cash.

## Timing
Refunds are issued within 5 business days of the returned item being received at
the warehouse. Store credit is issued immediately on inspection.

## Exclusions
Consumable accessories, opened software, and clearance items marked FINAL SALE are
not returnable under any circumstances.""",
    },
    "warranty-cx4400": {
        "title": "CX-4400 Limited Warranty",
        "uri": "https://contoso.example/warranty/cx-4400",
        "text": """## Coverage
The CX-4400 industrial controller carries a 36-month limited warranty from the date
of shipment, covering defects in materials and workmanship.

## Exclusions
The warranty does not cover damage from operation outside the rated temperature
range of -10C to 55C, from unauthorised firmware, or from water ingress.

## Claims
Warranty claims must be filed within 14 days of the fault being discovered. A
returned-materials authorisation number is required before shipping any unit back.""",
    },
    "shipping": {
        "title": "Shipping and Delivery",
        "uri": "https://contoso.example/policies/shipping",
        "text": """## Domestic
Standard delivery is 3 to 5 business days. Express delivery is next business day if
ordered before 14:00 local time.

## International
International orders ship DDU. The recipient is responsible for duties and import
taxes. Delivery is 7 to 21 business days depending on customs.

## Damaged in transit
Report transit damage within 48 hours with photographs. Replacements ship at no
cost and do not consume the returns window.""",
    },
    "escalation": {
        "title": "Support Escalation Matrix",
        "uri": "https://contoso.example/support/escalation",
        "text": """## Severity levels
S1 is a total production outage and carries a 1-hour response target. S2 is degraded
operation with a 4-hour target. S3 is a question or minor issue with a 1-business-day
target.

## Escalation path
Tier 1 handles S3. S2 escalates to Tier 2 after 4 hours without resolution. S1 pages
the duty engineer immediately and notifies the account manager.

## Refund authority
Tier 1 may authorise store credit up to 200 USD. Anything above that, or any cash
refund outside policy, requires a manager.""",
    },
}

print(f"{len(DOCS)} documents, {sum(len(d['text']) for d in DOCS.values())} characters")

## 2. Chunking

Three strategies over the same text. Watch two things: how many chunks each
produces (that is your index size and your embedding bill), and whether a chunk
still makes sense on its own (that is your retrieval quality).

The critical trick is in `structural_chunks`: every chunk carries its **document
title and section heading**. A chunk that reads *"must be filed within 14 days"* is
unretrievable and uncitable. *"CX-4400 Limited Warranty → Claims → must be filed
within 14 days"* is both.

In [ ]:
import re


def fixed_chunks(text, size=400, overlap=60):
    """Naive baseline: fixed character windows. Fast, and splits mid-sentence."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]


def paragraph_chunks(text, budget=500):
    """Pack whole paragraphs up to a character budget. Never splits a sentence."""
    out, buf = [], ""
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        if buf and len(buf) + len(para) > budget:
            out.append(buf)
            buf = para
        else:
            buf = f"{buf}\n\n{para}" if buf else para
    if buf:
        out.append(buf)
    return out


def structural_chunks(doc_id, doc):
    """Split on markdown headings and prepend title + heading to every chunk.

    This is the version we actually index.
    """
    parts = re.split(r"^## ", doc["text"], flags=re.MULTILINE)
    chunks = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        heading, _, body = part.partition("\n")
        body = body.strip()
        if not body:
            continue
        chunks.append({
            "doc_id": doc_id,
            "title": doc["title"],
            "section": heading.strip(),
            "uri": doc["uri"],
            # The contextual prefix is part of the EMBEDDED text, not just metadata.
            "content": f"{doc['title']} > {heading.strip()}\n{body}",
        })
    return chunks


sample = DOCS["returns-policy"]["text"]
print(f"fixed(400/60)  -> {len(fixed_chunks(sample)):>2} chunks")
print(f"paragraph(500) -> {len(paragraph_chunks(sample)):>2} chunks")
print(f"structural     -> {len(structural_chunks('returns-policy', DOCS['returns-policy'])):>2} chunks")

print("\nfixed chunk 1 ends mid-thought:")
print("  ..." + fixed_chunks(sample)[0][-90:].replace("\n", " "))
print("\nstructural chunk 1 stands alone:")
print("  " + structural_chunks("returns-policy", DOCS["returns-policy"])[0]["content"][:150].replace("\n", " "))

In [ ]:
chunks = []
for doc_id, doc in DOCS.items():
    chunks.extend(structural_chunks(doc_id, doc))

for i, c in enumerate(chunks):
    c["id"] = f"{c['doc_id']}-{i}"

print(f"{len(chunks)} chunks to index")
for c in chunks[:4]:
    print(f"  {c['id']:<20} {c['title']} > {c['section']}")

## 3. Create the index

Four things go into this schema, and the exam can ask about each:

| Element | Why |
|---|---|
| `SearchableField` on `content` | enables BM25 keyword matching |
| `SearchField` with `vector_search_dimensions` + profile | enables vector matching |
| `VectorSearch` (HNSW algorithm + profile) | how neighbours are found |
| `SemanticSearch` configuration | which fields the reranker reads |

`vector_search_dimensions` **must** match the embedding model: 1536 for
`text-embedding-3-small`, 3072 for `-large`. A mismatch is the single most common
cause of "vector search returns nothing while keyword works".

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, VectorSearchProfile, HnswAlgorithmConfiguration, HnswParameters,
    VectorSearchAlgorithmMetric,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField,
)

DIMS = 1536   # text-embedding-3-small

index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        # filterable/facetable so we can scope retrieval later without re-embedding
        SimpleField(name="doc_id", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="section", type=SearchFieldDataType.String),
        SimpleField(name="uri", type=SearchFieldDataType.String, retrievable=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=DIMS,
            vector_search_profile_name="hnsw-profile",
        ),
    ],
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw-config",
                parameters=HnswParameters(
                    m=4,                    # graph connectivity: higher = better recall, bigger index
                    ef_construction=400,    # build-time breadth
                    ef_search=500,          # query-time breadth
                    metric=VectorSearchAlgorithmMetric.COSINE,
                ),
            )
        ],
        profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-config")],
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="default",
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")],
                    keywords_fields=[SemanticField(field_name="section")],
                ),
            )
        ],
    ),
)

idx_client = search_index_client()
idx_client.create_or_update_index(index)
print("index created:", INDEX_NAME)

## 4. Embed and upload

We embed in our own code here so the mechanics are visible. In production you would
usually use **integrated vectorization** — attach a vectorizer to the index and a
`SplitSkill` + `AzureOpenAIEmbeddingSkill` skillset to an indexer, and Search does
the chunking and embedding on a schedule. Same result, less code, less control.

In [ ]:
import time

vectors = embed([c["content"] for c in chunks])
for c, v in zip(chunks, vectors):
    c["content_vector"] = v

sc = search_client(INDEX_NAME)
result = sc.upload_documents(documents=chunks)
print(f"uploaded {sum(1 for r in result if r.succeeded)}/{len(chunks)} chunks")

time.sleep(3)   # indexing is near-real-time but not instantaneous
print("documents in index:", sc.get_document_count())

## 5. Four retrieval modes, one query

Two deliberately different queries:

- **`"CX-4400"`** — a literal part number. Keyword should dominate; pure vector
  search struggles with rare tokens like this.
- **`"can I get my money back after unwrapping it"`** — pure paraphrase. The word
  "refund" never appears, so keyword struggles and vector should win.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def show(label, results):
    print(f"\n{label}")
    print("-" * 74)
    any_row = False
    for r in results:
        any_row = True
        rr = r.get("@search.reranker_score")
        extra = f"  reranker={rr:.2f}" if rr is not None else ""
        print(f"  {r['@search.score']:>7.3f}{extra}  {r['title']} > {r['section']}")
    if not any_row:
        print("  (no results)")


def keyword(q, k=3):
    return sc.search(search_text=q, top=k, select=["title", "section", "content", "uri", "id"])


def vector(q, k=3):
    vq = VectorizedQuery(vector=embed(q)[0], k_nearest_neighbors=k, fields="content_vector")
    return sc.search(search_text=None, vector_queries=[vq], top=k,
                     select=["title", "section", "content", "uri", "id"])


def hybrid(q, k=3):
    """Keyword AND vector in one request. Search fuses the two ranked lists with RRF."""
    vq = VectorizedQuery(vector=embed(q)[0], k_nearest_neighbors=k * 3, fields="content_vector")
    return sc.search(search_text=q, vector_queries=[vq], top=k,
                     select=["title", "section", "content", "uri", "id"])


def hybrid_semantic(q, k=3):
    """Hybrid, then a Microsoft cross-encoder re-ranks the top ~50 candidates."""
    vq = VectorizedQuery(vector=embed(q)[0], k_nearest_neighbors=k * 3, fields="content_vector")
    return sc.search(
        search_text=q, vector_queries=[vq], top=k,
        query_type="semantic", semantic_configuration_name="default",
        query_caption="extractive", query_answer="extractive",
        select=["title", "section", "content", "uri", "id"],
    )


LITERAL = "CX-4400 temperature range"
show(f"KEYWORD  '{LITERAL}'", keyword(LITERAL))
show(f"VECTOR   '{LITERAL}'", vector(LITERAL))

In [ ]:
PARAPHRASE = "can I get my money back after unwrapping it"

show(f"KEYWORD  '{PARAPHRASE}'", keyword(PARAPHRASE))
show(f"VECTOR   '{PARAPHRASE}'", vector(PARAPHRASE))
show(f"HYBRID   '{PARAPHRASE}'", hybrid(PARAPHRASE))

try:
    show(f"HYBRID+SEMANTIC '{PARAPHRASE}'", hybrid_semantic(PARAPHRASE))
except Exception as e:
    print("\nsemantic ranker unavailable on this service/tier:", str(e)[:160])

Note the score columns. `@search.score` from a **hybrid** query is an **RRF** score,
not a BM25 score and not a cosine similarity — it is a rank-fusion number, typically
around 0.01–0.03, and it is not comparable to the BM25 score from a keyword-only
query. `@search.reranker_score` is a separate 0–4 relevance score from the semantic
cross-encoder.

> **Exam note.** Three symptoms, three fixes. *Exact IDs return nothing* → you are
> vector-only, add keyword. *Paraphrased questions return nothing* → you are
> keyword-only, add vectors. *Right documents, wrong order* → add the semantic
> ranker. Hybrid + semantic is the default recommendation because it covers all
> three.

## 6. Ground the generation and cite

Grounding is prompt discipline plus a data structure. Numbered chunks in, numbered
citations out, and a hard escape hatch (`NOT_IN_CONTEXT`) so the model has a legal
way to admit it does not know.

In [ ]:
GROUNDED_SYSTEM = """You answer strictly from the numbered CONTEXT below.

Rules:
- Use ONLY facts present in the CONTEXT. Never use prior knowledge.
- Cite every factual claim inline as [n], matching the context numbering.
- If the CONTEXT does not contain the answer, reply exactly: NOT_IN_CONTEXT
- Be concise."""


def retrieve(question, k=4):
    """Hybrid + semantic where available, plain hybrid otherwise."""
    try:
        return list(hybrid_semantic(question, k))
    except Exception:
        return list(hybrid(question, k))


def build_context(hits):
    """Numbered context block + a parallel citation table for attribution."""
    lines, sources = [], []
    for n, h in enumerate(hits, start=1):
        lines.append(f"[{n}] {h['content']}")
        sources.append({"n": n, "id": h["id"], "title": h["title"],
                        "section": h["section"], "uri": h["uri"]})
    return "\n\n".join(lines), sources


def rag(question, k=4):
    hits = retrieve(question, k)
    context, sources = build_context(hits)
    resp = chat_client().chat.completions.create(
        model=cfg["MODEL_MINI"],
        messages=[
            {"role": "system", "content": GROUNDED_SYSTEM},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0,
    )
    return {
        "question": question,
        "answer": resp.choices[0].message.content,
        "context": context,
        "sources": sources,
        "usage": resp.usage,
    }


out = rag("I opened the box. Can I get my money back, and how fast?")
print(out["answer"])
print("\nCitations")
for s in out["sources"]:
    print(f"  [{s['n']}] {s['title']} > {s['section']}  {s['uri']}")
print()
show_usage(out)

In [ ]:
# The escape hatch matters as much as the answer. Ask something the corpus
# genuinely does not cover.
miss = rag("What is Contoso's parental leave allowance?")
print(miss["answer"])
print("\nRefused correctly:", miss["answer"].strip().startswith("NOT_IN_CONTEXT"))

Without that instruction the model would happily invent a leave policy — fluent,
plausible, and completely fabricated. A RAG system without a refusal path is a
hallucination machine with extra steps.

## 7. Prove it, with evaluators

Two different questions, two different evaluators:

- `GroundednessEvaluator` — *is the answer supported by what we retrieved?*
- `RetrievalEvaluator` — *did we retrieve the right things in the right order?*

The second is the one people forget, and it is where most RAG bugs actually live.

In [ ]:
from azure.ai.evaluation import GroundednessEvaluator, RelevanceEvaluator, RetrievalEvaluator

model_config = {
    "azure_endpoint": cfg["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": cfg["MODEL_MINI"],
    "api_version": cfg.get("AZURE_OPENAI_API_VERSION", "2025-04-01-preview"),
}
groundedness = GroundednessEvaluator(model_config)
relevance = RelevanceEvaluator(model_config)
retrieval = RetrievalEvaluator(model_config)

QUESTIONS = [
    "I opened the box. Can I get my money back, and how fast?",
    "What temperature range voids the CX-4400 warranty?",
    "How long do I have to report a package that arrived crushed?",
    "Can a Tier 1 agent approve a 500 dollar store credit?",
]

records = []
for q in QUESTIONS:
    r = rag(q)
    g = groundedness(query=q, response=r["answer"], context=r["context"])
    rel = relevance(query=q, response=r["answer"])
    ret = retrieval(query=q, context=r["context"])
    records.append({**r, "groundedness": g["groundedness"],
                    "relevance": rel["relevance"], "retrieval": ret["retrieval"]})
    print(f"g={g['groundedness']}  rel={rel['relevance']}  ret={ret['retrieval']}  | {q}")
    print(f"    {r['answer'][:150]}")

> **Exam note.** Low **retrieval** with high **groundedness** means the model was
> faithful to bad context — fix the index or the query, not the prompt. High
> retrieval with low groundedness means the right documents were there and the
> model ignored them — fix the prompt or the model. Diagnosing RAG without both
> numbers is guesswork.

## 8. Multistep reasoning

Naive RAG embeds the literal user text. That breaks the moment a user says
"and what about that one?" — the pronoun carries no retrievable signal.

Four steps, cheapest first: **rewrite → decompose → retrieve per subquery →
synthesise → self-check.**

In [ ]:
import json


def rewrite(history, follow_up):
    """Turn a context-dependent follow-up into a standalone, searchable query."""
    turns = "\n".join(f"{m['role']}: {m['content']}" for m in history)
    return ask(
        f"CONVERSATION:\n{turns}\n\nFOLLOW-UP: {follow_up}\n\n"
        "Rewrite the follow-up as a single standalone search query that needs no "
        "conversation history. Output only the query.",
        temperature=0,
    ).strip()


history = [
    {"role": "user", "content": "How long is the CX-4400 warranty?"},
    {"role": "assistant", "content": "36 months from date of shipment."},
]
follow_up = "and how quickly do I have to tell you about a fault?"

standalone = rewrite(history, follow_up)
print("raw follow-up :", follow_up)
print("rewritten     :", standalone)

print("\nretrieval on the RAW text:")
show("", hybrid(follow_up, 2))
print("retrieval on the REWRITTEN text:")
show("", hybrid(standalone, 2))

In [ ]:
def decompose(question, max_parts=3):
    """Split a multi-part question into independently searchable subqueries."""
    raw = chat_client().chat.completions.create(
        model=cfg["MODEL_MINI"],
        messages=[
            {"role": "system", "content":
             "Split the question into independent search queries. If it is already "
             "a single query, return just that one."},
            {"role": "user", "content": question},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "subqueries", "strict": True,
                "schema": {
                    "type": "object", "additionalProperties": False,
                    "properties": {"queries": {"type": "array", "items": {"type": "string"}}},
                    "required": ["queries"],
                },
            },
        },
        temperature=0,
    )
    return json.loads(raw.choices[0].message.content)["queries"][:max_parts]


MULTI = (
    "My CX-4400 died after 20 months in a 60C room and the box was already open — "
    "am I covered, and who can sign off a refund over 200 dollars?"
)
subs = decompose(MULTI)
for s in subs:
    print(" -", s)

In [ ]:
def multistep_rag(question, k=3):
    """decompose -> retrieve per subquery -> dedupe -> generate -> self-check."""
    subqueries = decompose(question)

    seen, merged = set(), []
    for sq in subqueries:
        for hit in retrieve(sq, k):
            if hit["id"] not in seen:
                seen.add(hit["id"])
                merged.append(hit)

    context, sources = build_context(merged)
    answer = chat_client().chat.completions.create(
        model=cfg["MODEL_MINI"],
        messages=[
            {"role": "system", "content": GROUNDED_SYSTEM},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0,
    ).choices[0].message.content

    # Self-critique: a second, cheap call whose only job is to find unsupported claims.
    critique = chat_client().chat.completions.create(
        model=cfg["MODEL_MINI"],
        messages=[{"role": "user", "content":
                   f"CONTEXT:\n{context}\n\nANSWER:\n{answer}\n\n"
                   "List any claim in the ANSWER not supported by the CONTEXT."}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "critique", "strict": True,
                "schema": {
                    "type": "object", "additionalProperties": False,
                    "properties": {
                        "fully_grounded": {"type": "boolean"},
                        "unsupported_claims": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": ["fully_grounded", "unsupported_claims"],
                },
            },
        },
        temperature=0,
    ).choices[0].message.content

    return {"subqueries": subqueries, "answer": answer, "sources": sources,
            "critique": json.loads(critique), "chunks_used": len(merged)}


res = multistep_rag(MULTI)
print("subqueries  :", res["subqueries"])
print("chunks used :", res["chunks_used"])
print("\nANSWER\n", res["answer"])
print("\nSELF-CHECK\n", json.dumps(res["critique"], indent=2))
print("\nSOURCES")
for s in res["sources"]:
    print(f"  [{s['n']}] {s['title']} > {s['section']}")

That is a **tool-augmented, multistep reasoning pipeline**: a planning step, N tool
invocations, a synthesis step, and a verification step, each individually traceable,
evaluable, and costed. Compare it with the single `rag()` call — same corpus, but the
multi-part question now gets answered on evidence from two different documents.

## 9. "On Your Data" — the same thing in one call

Everything above, done by the service. You give up chunking, prompt, and filter
control; you get a `citations` array for free.

This requires the Foundry resource's managed identity to hold *Search Index Data
Reader* on the Search service. If that role assignment is missing the cell reports
the error rather than failing the notebook.

In [ ]:
try:
    oyd = chat_client().chat.completions.create(
        model=cfg["MODEL_MINI"],
        messages=[{"role": "user", "content": "Can I refund an opened item, and how long does it take?"}],
        extra_body={
            "data_sources": [{
                "type": "azure_search",
                "parameters": {
                    "endpoint": cfg["AZURE_SEARCH_ENDPOINT"],
                    "index_name": INDEX_NAME,
                    "authentication": {"type": "system_assigned_managed_identity"},
                    "query_type": "vector_simple_hybrid",
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": cfg["MODEL_EMBEDDING"],
                    },
                    "fields_mapping": {
                        "content_fields": ["content"],
                        "title_field": "title",
                        "url_field": "uri",
                        "vector_fields": ["content_vector"],
                    },
                    "in_scope": True,     # refuse questions the index cannot answer
                    "strictness": 3,      # 1-5; higher = fewer weakly-grounded answers
                    "top_n_documents": 4,
                },
            }]
        },
        temperature=0,
    )
    msg = oyd.choices[0].message
    print(msg.content)
    ctx = getattr(msg, "context", None) or {}
    for c in ctx.get("citations", []):
        print(f"  - {c.get('title')}  {c.get('url')}")
except Exception as e:
    print("On Your Data unavailable:", type(e).__name__)
    print(str(e)[:400])
    print("\nMost common cause: the Foundry resource's managed identity lacks")
    print("'Search Index Data Reader' on the Search service.")

| | Custom RAG (sections 5–8) | On Your Data (section 9) | Agentic retrieval (unit 02.3) |
|---|---|---|---|
| Who chunks | you | the ingestion wizard | the ingestion wizard |
| Who writes the prompt | you | the service | the service |
| Who decides *whether* to search | you, always | the service, always | **the model**, per turn |
| Multi-hop | only if you build it | no | yes |
| Citations | you build them | `context.citations` | message annotations |
| Best for | control, unusual logic | shipping fast | conversational, multi-hop |

## 10. Cleanup — do not skip this

Deletes the index created by this notebook. **This does not stop the Search service
billing.** If you provisioned a Basic (or higher) tier service just for this course,
delete the service itself, or run `99_teardown`.

In [ ]:
try:
    search_index_client().delete_index(INDEX_NAME)
    print("deleted index:", INDEX_NAME)
except Exception as e:
    print("index delete failed (may already be gone):", type(e).__name__, str(e)[:150])

print("\nRemaining indexes on the service:")
for name in search_index_client().list_index_names():
    print("  ", name)

print("\nReminder: the SEARCH SERVICE still bills hourly. Deleting an index does not")
print("stop that. See 99_teardown to delete the service or the resource group.")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Chunk size sweep.** Re-index the corpus using `fixed_chunks` at sizes 200,
   600, and 1500 characters (rebuild the index each time under a different name,
   and delete each one). Run the four evaluation questions against each and record
   `retrieval` and `groundedness`. Which size wins, and what happens to
   `prompt_tokens`?
2. **Strip the context prefix.** Re-index with `content` set to the raw section
   body only — no `"{title} > {heading}"` prefix. Rerun the four questions. How much
   does retrieval quality drop, and can you still build a citation?
3. **Filtered retrieval.** Add `filter="doc_id eq 'warranty-cx4400'"` to the hybrid
   query and ask a returns question. What does the grounded answer do, and which
   real-world requirement (multi-tenancy, document-level permissions) does this
   pattern implement?
4. **Strictness.** Run the On Your Data cell with `strictness=1` and then `5`,
   asking a question that is only weakly covered. Describe the trade-off you just
   configured.

In [ ]:
# Your work here.